# Running analysis of 211117_SunTag_Lamp1Apple_d10 data

In [1]:
import nd2
import matplotlib.pyplot as plt
import numpy as np
import os
import skimage as ski
import pandas as pd
import napari
import trackpy as tp

# Working directory set up

In [2]:
from pathlib import Path
import nd2
import pandas as pd

metadata = {}

# Paths and parameters
images_dir = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8")
images = list(sorted(images_dir.glob('*.nd2')))

In [3]:
print(images)

[PosixPath('/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8/RiboTagI76_30snd_d8_1001.nd2'), PosixPath('/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8/RiboTagI76_30snd_d8_1002.nd2'), PosixPath('/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8/RiboTagI76_30snd_d8_1003.nd2'), PosixPath('/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8/RiboTagI76_30snd_d8_1004.nd2'), PosixPath('/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8/RiboTagI76_30snd_d8_1005.nd2')]


# Set up output folders

In [4]:
output_dir = "output_20250613"
os.makedirs(output_dir, exist_ok=True)

overlay_output_dir = "output_20250613/overlays"
os.makedirs(overlay_output_dir, exist_ok=True)

# Set up functions

## Capture metadata

In [5]:
from inspect import signature, getcallargs

def capture_metadata(metadata_store, prefix=None):
    def decorator(func):
        def wrapper(*args, **kwargs):
            bound_args = getcallargs(func, *args, **kwargs)
            tag = f"{prefix}_" if prefix else ""
            for key, val in bound_args.items():
                metadata_store[f"{tag}{key}"] = val
            return func(*args, **kwargs)
        return wrapper
    return decorator

## Image pre-processing and masking function

In [6]:
import numpy as np
from scipy.ndimage import median_filter
from skimage import filters, morphology
from scipy import ndimage as ndi
from skimage.morphology import disk

@capture_metadata(metadata, prefix="preprocess")
def preprocess_stack_with_mask(img_stack, 
                               blob_channel=2, 
                               mask_channel=1, 
                               median_filter_size=10, 
                               min_object_size=400, 
                               dilation_radius=5,
                               apply_mask=True):
    """
    Preprocess a 3D image stack by background subtraction and optionally mask with a binary region
    generated from another channel.

    Parameters:
        img_stack (np.ndarray): Input image stack of shape (t, c, y, x).
        blob_channel (int): The index of the channel to process.
        mask_channel (int): The index of the channel used for creating a binary mask.
        median_filter_size (int): Size of the median filter for background estimation.
        min_object_size (int): Minimum size of objects to keep in the binary mask.
        dilation_radius (int): Radius for morphological dilation of the binary mask.
        apply_mask (bool): Whether to apply the binary mask to the preprocessed output.

    Returns:
        np.ndarray: Preprocessed (and optionally masked) image stack of shape (t, y, x).
    """
    # Mean projection for background subtraction
    mean_proj = np.mean(img_stack[:, blob_channel], axis=0)
    background = median_filter(mean_proj, size=median_filter_size)

    # Subtract background and clip negatives
    preprocessed_frames = [
        np.clip(frame - background, 0, None).astype(np.uint16)
        for frame in img_stack[:, blob_channel]
    ]
    preprocessed_stack = np.stack(preprocessed_frames)

    if apply_mask:
        # Create a mask from the max projection of the mask channel
        ch_mask_proj = np.max(img_stack[:, mask_channel], axis=0)
        thresh = filters.threshold_otsu(ch_mask_proj)
        binary = ch_mask_proj > thresh
        filled = ndi.binary_fill_holes(binary)
        large_objects_mask = morphology.remove_small_objects(filled, min_size=min_object_size)
        dilated = morphology.binary_dilation(large_objects_mask, disk(dilation_radius))

        # Apply the mask
        preprocessed_stack *= dilated.astype(np.uint16)

    return preprocessed_stack


## Identify blobs function

In [7]:
import dask
from dask import delayed, compute
import numpy as np
import pandas as pd
from skimage.feature import blob_log
from skimage import filters

@capture_metadata(metadata, prefix="detect")
def detect_blobs_in_stack(
    image_stack,
    threshold=0.12,
    min_sigma=1,
    max_sigma=3,
    gaussian_sigma=1,
    normalize=True
):
    """
    Detects blobs in each frame of a 3D image stack using Laplacian of Gaussian.

    Parameters:
        image_stack (np.ndarray): A 3D array of shape (t, y, x).
        threshold (float): Threshold for blob detection.
        min_sigma (float): Minimum standard deviation for Gaussian kernel.
        max_sigma (float): Maximum standard deviation for Gaussian kernel.
        gaussian_sigma (float): Sigma for initial Gaussian blur before blob detection. # i think this mucks with blob detection too much
        normalize (bool): Whether to normalize each frame before processing.

    Returns:
        pd.DataFrame: DataFrame with columns ['frame', 'x', 'y', 'size'].
    """

    @delayed
    def process_frame(t, frame):
        if normalize:
            frame = (frame - np.min(frame)) / (np.max(frame) - np.min(frame) + 1e-8)
        #blurred = filters.gaussian(frame, sigma=gaussian_sigma)
        blobs = blob_log(frame, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
        return [{'frame': t, 'x': x, 'y': y, 'size': sigma * np.sqrt(2)} for y, x, sigma in blobs]

    tasks = [process_frame(t, frame) for t, frame in enumerate(image_stack)]
    results = compute(*tasks)
    positions = [entry for sublist in results for entry in sublist]

    print(f"Puncta counting complete: {len(positions)} total puncta were found")
    return pd.DataFrame(positions)


## Process blobs function
Removes blobs that linger in an area too long

In [8]:
import pandas as pd

@capture_metadata(metadata, prefix="filter")
def filter_dense_blobs(blob_df, bin_size=5, blob_filter=10):
    """
    Filters out blobs that appear too frequently within a very small local neighborhood.

    Parameters:
        blob_df (pd.DataFrame): DataFrame with 'x' and 'y' columns of blob coordinates.
        bin_size (int): Pixel binning size for rounding positions.
        blob_filter (int): Maximum allowed count of blobs in a bin before filtering them out.

    Returns:
        pd.DataFrame: Filtered DataFrame with blobs below the density threshold.
    """
    rounded = blob_df.copy()
    rounded['x_rounded'] = (rounded['x'] // bin_size * bin_size).astype(int)
    rounded['y_rounded'] = (rounded['y'] // bin_size * bin_size).astype(int)

    counts = rounded.groupby(['x_rounded', 'y_rounded']).size().reset_index(name='count')
    merged = rounded.merge(counts, on=['x_rounded', 'y_rounded'])

    filtered_blobs = merged[merged['count'] < blob_filter].reset_index(drop=True)

    print(f"Puncta filtering complete: {len(filtered_blobs)} puncta remain")
    return filtered_blobs


## Kalman filter for particle linking function

In [9]:
from filterpy.kalman import KalmanFilter
from scipy.spatial import cKDTree

class KalmanTrack:
    def __init__(self, initial_detection, particle_id, frame, max_missed=5):
        x, y = initial_detection
        self.kf = KalmanFilter(dim_x=4, dim_z=2)
        dt = 1
        self.kf.F = np.array([[1, 0, dt, 0],
                              [0, 1, 0, dt],
                              [0, 0, 1, 0],
                              [0, 0, 0, 1]])
        self.kf.H = np.array([[1, 0, 0, 0],
                              [0, 1, 0, 0]])
        self.kf.R *= 5
        self.kf.P *= 1000
        self.kf.Q = np.eye(4) * 0.1
        self.kf.x = np.array([x, y, 0, 0])

        self.id = particle_id
        self.history = [{'frame': frame, 'x': x, 'y': y, 'particle': self.id}]
        self.missed = 0
        self.max_missed = max_missed

    def predict(self):
        self.kf.predict()
        return self.kf.x[:2]

    def update(self, detection, frame):
        self.kf.update(np.array(detection))
        x, y = self.kf.x[:2]
        self.history.append({'frame': frame, 'x': x, 'y': y, 'particle': self.id})
        self.missed = 0

    def mark_missed(self):
        self.missed += 1

    def is_dead(self):
        return self.missed > self.max_missed

@capture_metadata(metadata, prefix="kalman")
def kalman_track_blobs(all_blobs, max_distance=15, max_missed=5):
    active_tracks = []
    finished_tracks = []
    next_id = 0

    for frame, detections in all_blobs.groupby('frame'):
        detected_positions = detections[['x', 'y']].to_numpy()

        # Predict positions for all current tracks
        predictions = [track.predict() for track in active_tracks]
        assigned = set()

        if len(predictions) > 0 and len(detected_positions) > 0:
            tree = cKDTree(detected_positions)
            for i, pred in enumerate(predictions):
                dist, idx = tree.query(pred, distance_upper_bound=max_distance)
                if idx != len(detected_positions) and idx not in assigned:
                    active_tracks[i].update(detected_positions[idx], frame)
                    assigned.add(idx)
                else:
                    active_tracks[i].mark_missed()
        else:
            for track in active_tracks:
                track.mark_missed()

        # Start new tracks for unassigned detections
        for i, pos in enumerate(detected_positions):
            if i not in assigned:
                new_track = KalmanTrack(pos, next_id, frame, max_missed)
                active_tracks.append(new_track)
                next_id += 1

        # Remove dead tracks
        still_alive = []
        for track in active_tracks:
            if track.is_dead():
                finished_tracks.append(track)
            else:
                still_alive.append(track)
        active_tracks = still_alive

    # Add remaining active tracks to finished
    finished_tracks.extend(active_tracks)

    # Combine history
    all_tracks = [row for track in finished_tracks for row in track.history]        
    all_tracks = pd.DataFrame(all_tracks)
    print(f"Particle tracking complete: There are {len(all_tracks.groupby('particle'))} unique particle tracks")
    return all_tracks


## Track cleanup functions

In [10]:
import numpy as np
import pandas as pd
import trackpy as tp  # assuming you're using trackpy
from shapely.geometry import LineString

@capture_metadata(metadata, prefix="displacement")
def filter_tracks_by_net_displacement(tracks_df, displacement_threshold=50, min_frames=10):
    """
    Filter tracks by net displacement after removing short tracks.

    Parameters:
        tracks_df (pd.DataFrame): DataFrame with ['particle', 'x', 'y'].
        displacement_threshold (float): Minimum net displacement required to keep a track.
        min_frames (int): Minimum number of frames a track must appear in.

    Returns:
        pd.DataFrame: Filtered tracking DataFrame.
    """
    # Remove short tracks
    filtered_tracks = tp.filter_stubs(tracks_df, threshold=min_frames).reset_index(drop=True)
    print(f"Track stub removal complete: {len(filtered_tracks.groupby('particle'))} unique particle tracks")

    # Compute net displacement
    displacements = []
    for pid, group in filtered_tracks.groupby('particle'):
        start = group.iloc[0][['x', 'y']].values
        end = group.iloc[-1][['x', 'y']].values
        net_disp = np.linalg.norm(end - start)
        displacements.append({'particle': pid, 'net_displacement': net_disp})

    displacement_df = pd.DataFrame(displacements)

    # Filter by displacement
    valid_particles = displacement_df.loc[displacement_df['net_displacement'] > displacement_threshold, 'particle']
    print(f"Track filtering complete: {len(valid_particles)} unique particle tracks")

    return filtered_tracks[filtered_tracks['particle'].isin(valid_particles)].reset_index(drop=True)

from shapely.geometry import LineString

def count_self_intersections(track_df):
    """
    For each particle track, count the number of self-intersections.
    
    Parameters:
        track_df (pd.DataFrame): Must include ['particle', 'x', 'y'].
    
    Returns:
        pd.DataFrame: Columns ['particle', 'self_intersections'].
    """
    results = []

    for pid, group in track_df.groupby('particle'):
        coords = group[['x', 'y']].values

        if len(coords) < 2:
            # Not enough points to form a LineString
            results.append({'particle': pid, 'self_intersections': 0})
            continue

        line = LineString(coords)

        # Count self-intersections via segment crossing
        n_self_intersections = 0
        segments = [LineString([coords[i], coords[i+1]]) for i in range(len(coords) - 1)]

        for i, seg1 in enumerate(segments):
            for j in range(i + 2, len(segments)):  # skip adjacent or same segment
                if seg1.crosses(segments[j]):
                    n_self_intersections += 1

        results.append({'particle': pid, 'self_intersections': n_self_intersections})

    return pd.DataFrame(results)

@capture_metadata(metadata, prefix="self_intersections")
def filter_tracks_by_intersection_count(tracks_df, max_crossings=10):
    """
    Keeps only tracks with self-intersections <= max_crossings.

    Parameters:
        tracks_df (DataFrame): Tracking DataFrame with ['particle', 'x', 'y'].
        max_crossings (int): Maximum number of allowed crossovers.

    Returns:
        DataFrame: Filtered tracking DataFrame.
    """
    # Compute self-intersection counts for all tracks
    intersection_df = count_self_intersections(tracks_df)

    # Get list of particles with intersection count ≤ threshold
    valid_particles = intersection_df.loc[
        intersection_df['self_intersections'] <= max_crossings, 'particle'
    ]

    print(f"Track self-intersection filtering complete: {len(valid_particles)} tracks with ≤ {max_crossings} crossings")
    return tracks_df[tracks_df['particle'].isin(valid_particles)].reset_index(drop=True)

## Generate movies of individual tracks

In [11]:
import cv2
import os
import numpy as np

def normalize_crop_percentile(crop, lower=2, upper=98):
    """Robust per-channel percentile-based normalization."""
    T, C, h, w = crop.shape
    crop_norm = np.zeros_like(crop, dtype=np.uint8)

    for c in range(C):
        channel_data = crop[:, c].flatten()
        p_low, p_high = np.percentile(channel_data, [lower, upper])

        if p_high - p_low < 1e-3:
            # No dynamic range, likely blank — skip normalization
            crop_norm[:, c] = 0
        else:
            norm = (crop[:, c] - p_low) / (p_high - p_low)
            norm = np.clip(norm * 255, 0, 255)
            crop_norm[:, c] = norm.astype(np.uint8)

    return crop_norm

def save_track_movies(img_array, track_df, image_name, output_dir, padding=40, fps=10):
    """
    Save cropped MP4 movies centered around each track from a 4D image array.

    Parameters:
        img_array (ndarray): Shape (T, C, Y, X) numpy array.
        track_df (DataFrame): DataFrame with ['particle', 'frame', 'y', 'x'].
        image_name (str): Name of the image file being processed (e.g., 'img_1031.nd2').
        output_dir (str): Directory to save MP4 files.
        padding (int): Padding around the bounding box of each track.
        fps (int): Frames per second for output videos.
    """
    os.makedirs(output_dir, exist_ok=True)
    T, C, Y, X = img_array.shape

    # Remove file extension and replace problematic characters
    base_name = os.path.splitext(image_name)[0].replace(" ", "_")

    for pid, group in track_df.groupby('particle'):
        y_min, y_max = int(group['y'].min()), int(group['y'].max())
        x_min, x_max = int(group['x'].min()), int(group['x'].max())

        y1 = max(y_min - padding, 0)
        y2 = min(y_max + padding, Y)
        x1 = max(x_min - padding, 0)
        x2 = min(x_max + padding, X)

        if y2 <= y1 or x2 <= x1:
            continue

        # Crop and normalize to 8-bit
        crop = img_array[:, :, y1:y2, x1:x2]  # shape: (T, C, h, w)
        crop_norm = normalize_crop_percentile(crop)

        for c in range(C):
            out_filename = f"{base_name}_track{pid}_ch{c}.mp4"
            out_path = os.path.join(output_dir, out_filename)
            h, w = crop.shape[2], crop.shape[3]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h), isColor=True)

            for t in range(T):
                frame = crop_norm[t, c, :, :]  # shape (h, w)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
                writer.write(frame_rgb)

            writer.release()
            print(f"Saved: {out_path}")


In [12]:
import cv2
import os
import numpy as np

def normalize_crop_percentile(crop, lower=2, upper=98):
    """Robust per-channel percentile-based normalization."""
    T, C, h, w = crop.shape
    crop_norm = np.zeros_like(crop, dtype=np.uint8)

    for c in range(C):
        channel_data = crop[:, c].flatten()
        p_low, p_high = np.percentile(channel_data, [lower, upper])

        if p_high - p_low < 1e-3:
            # No dynamic range, likely blank — skip normalization
            crop_norm[:, c] = 0
        else:
            norm = (crop[:, c] - p_low) / (p_high - p_low)
            norm = np.clip(norm * 255, 0, 255)
            crop_norm[:, c] = norm.astype(np.uint8)

    return crop_norm

def save_track_movies_with_overlay(img_array, track_df, image_name, output_dir=output_dir, padding=40, fps=10, spot_radius=5):
    """
    Save cropped MP4 movies with overlaid particle tracks and spots.

    Parameters:
        img_array (ndarray): Shape (T, C, Y, X).
        track_df (DataFrame): Must include ['particle', 'frame', 'y', 'x'].
        output_dir (str): Where to save MP4s.
        padding (int): Crop padding.
        fps (int): Video FPS.
        spot_radius (int): Radius of the current spot marker.
    """
    os.makedirs(output_dir, exist_ok=True)
    T, C, Y, X = img_array.shape

    # Remove file extension and replace problematic characters
    base_name = os.path.splitext(image_name)[0].replace(" ", "_")

    for pid, group in track_df.groupby('particle'):
        group = group.sort_values('frame')
        y_min, y_max = int(group['y'].min()), int(group['y'].max())
        x_min, x_max = int(group['x'].min()), int(group['x'].max())
        y1 = max(y_min - padding, 0)
        y2 = min(y_max + padding, Y)
        x1 = max(x_min - padding, 0)
        x2 = min(x_max + padding, X)

        if y2 <= y1 or x2 <= x1:
            continue

        crop = img_array[:, :, y1:y2, x1:x2]
        crop_norm = normalize_crop_percentile(crop)

        # Shift positions to local crop coordinates
        local_coords = group.copy()
        local_coords['y'] -= y1
        local_coords['x'] -= x1

        for c in range(C):
            out_filename = f"{base_name}_overlay_track{pid}_ch{c}.mp4"
            out_path = os.path.join(output_dir, out_filename)
            h, w = crop.shape[2], crop.shape[3]
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h), isColor=True)

            # Prepare path history
            path_points = []

            for t in range(T):
                frame_img = crop_norm[t, c, :, :]
                frame_bgr = cv2.cvtColor(frame_img, cv2.COLOR_GRAY2BGR)

                row = local_coords[local_coords['frame'] == t]
                if not row.empty:
                    cx, cy = int(row['x'].values[0]), int(row['y'].values[0])
                    path_points.append((cx, cy))

                    # Draw the current spot
                    cv2.circle(frame_bgr, (cx, cy), spot_radius, (0, 0, 255), 1)

                # Draw the path up to this point
                for i in range(1, len(path_points)):
                    cv2.line(frame_bgr, path_points[i - 1], path_points[i], (0, 255, 0), 1)

                writer.write(frame_bgr)

            writer.release()
            print(f"Saved: {out_path}")


# Data extraction

## Load image

In [13]:
from pathlib import Path

images_dir = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8")
images = list(sorted(images_dir.glob('*.nd2')))

image_index = 4

image_path = images[image_index]
print(image_path)

img_array = nd2.imread(image_path)

# add a dummy channel to the images
#img_array = img_array[:, np.newaxis, :, :]

/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/LysoAndMito/211012_RiboTagI76_d8/RiboTagI76_30snd_d8_1005.nd2


In [14]:
viewer = napari.Viewer()

viewer.add_image(img_array[:, 0], 
                 name='LYSOSOME')

viewer.add_image(img_array[:, 1], 
                 name='RIBOSOME')

<Image layer 'RIBOSOME' at 0x11303c690>

## Pre-process image
The this labels the image channels for prepprocessing and excecutes the preprocessing step. If apply_mask is True, this will generate a mask of the bright cell bodies from the SunTag channel. This will at least limit the number of puncta that are detected and will help speed up blob detection. Some of these images generate strange masks because some neurites cross over positive cells, so sometimes it is easier to just run the entire script on the whole image instead. The channel with puncta to be detected goes through a fair amount of processing. First, a mean projection is generated from the entire time series, then a median filter is applied to each frame of the time series from the mean projection, this drastically reduces the background fluorescense within the cell and improves blob detection. 

In [ ]:
preprocessed_stack = preprocess_stack_with_mask(img_array, 
                                                blob_channel=1, 
                                                mask_channel=1,
                                                apply_mask=False)


## Blob detection
This step is pretty slow and utilizes a Laplasican of Gaussian filter to detect blobs. I think this could probably be sped up with a DoG filter instead, but not sure its worth the effort right now. Probably the biggest factor to adjust in this whole pipeline is the threshold cutoff in this function. I am intersted in improving this by normalizing images a bit more before pre-processsing, but it has not been done yet. 

In [ ]:
all_RNA_blobs = detect_blobs_in_stack(preprocessed_stack, 
                                      threshold=0.0001, 
                                      min_sigma=1, 
                                      max_sigma=3, 
                                      gaussian_sigma=1,
                                      normalize=False) #slow

## Track linking and track clean-up

In [ ]:
RNA_blobs_filtered = filter_dense_blobs(all_RNA_blobs, 
                                        bin_size=5, 
                                        blob_filter=5)

tracks_df = kalman_track_blobs(RNA_blobs_filtered, 
                               max_distance=20, 
                               max_missed=3)

filtered_df = filter_tracks_by_net_displacement(tracks_df, 
                                                displacement_threshold=10, 
                                                min_frames=5)

filtered_2_df = filter_tracks_by_intersection_count(filtered_df)


## Metadata extraction

In [ ]:
import pandas as pd
import numpy as np

def annotate_tracks_df(tracks_df, metadata):
    for key, value in metadata.items():
        # Only allow scalar or string values; repeat for each row
        if np.isscalar(value) or isinstance(value, str):
            tracks_df[key] = value
        else:
            print(f"Skipping metadata field '{key}' — not a scalar")
    return tracks_df

metadata['image_name'] = image_path.name
metadata['image_index'] = image_index

all_tracks = annotate_tracks_df(filtered_2_df, metadata)

## Track visualization (napari)
Generally, this is the easiest way to validate good tracks, I just try and evaluate each track and confirm if it appropriately follows a particle. I'll note good tracks in the next chunk (good track list) to susbset the track dataframe. 

In [ ]:
tracks_data = all_tracks[['particle', 'frame', 'y', 'x']]
blobs_data = RNA_blobs_filtered[['frame', 'y', 'x']]

viewer = napari.Viewer()

viewer.add_image(img_array[:, 1], 
                 name='RIBOSOME')

viewer.add_image(preprocessed_stack, 
                 name='processed RIBOSOME channel')

viewer.add_image(img_array[:, 0], 
                 name='LYSOSOME')

track_layer = viewer.add_tracks(tracks_data,
                  name='Tracks')

track_layer.display_id = True

viewer.add_points(blobs_data,
                  name='blobs',  
                  size=30, 
                  face_color='transparent',
                  border_color='red',
                  border_width=0.1)

## Annotate validated tracks

In [ ]:
# annotate good tracks
good_track_list = [378, 3082, 2864, 107]

valid_tracks = all_tracks[all_tracks['particle'].isin(good_track_list)].copy()

valid_track_fn = f"{image_path.stem}_valid_tracks.csv"


In [ ]:
tracks_data = valid_tracks[['particle', 'frame', 'y', 'x']]
blobs_data = valid_tracks[['frame', 'y', 'x']]

viewer = napari.Viewer()

viewer.add_image(img_array[:, 0], 
                 name='RNA')

viewer.add_image(preprocessed_stack, 
                 name='processed RNA channel')

viewer.add_image(img_array[:, 1], 
                 name='Translation')

track_layer = viewer.add_tracks(tracks_data,
                  name='Good tracks')

track_layer.display_id = True

viewer.add_points(blobs_data,
                  name='blobs',  
                  size=30, 
                  face_color='transparent',
                  border_color='red',
                  border_width=0.1)

## Pull intensity values from puncta coordiates

In [ ]:
import numpy as np
from scipy.ndimage import map_coordinates
from scipy.stats import pearsonr

channels_to_measure = [0, 1]  

def manders_coeffs(chA, chB):
    chA = chA.flatten()
    chB = chB.flatten()

    # Calculate 90th percentile thresholds (top 10%)
    thresh1 = np.percentile(chA, 90)
    thresh2 = np.percentile(chB, 90)

    # Create binary masks for top 10% pixels
    mask1 = chA > thresh1
    mask2 = chB > thresh2

    # Avoid division by zero
    if np.sum(chA) == 0 or np.sum(chB) == 0:
        return np.nan, np.nan

    M1 = np.sum(chA[mask2]) / np.sum(chA)
    M2 = np.sum(chB[mask1]) / np.sum(chB)

    return M1, M2

def safe_pearsonr(a, b):
    a = a.flatten()
    b = b.flatten()
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return pearsonr(a, b)[0]

def extract_patch(img, y, x, size=10):
    half = size // 2
    y = int(round(y))
    x = int(round(x))
    y_min = max(y - half, 0)
    y_max = min(y + half + 1, img.shape[0])
    x_min = max(x - half, 0)
    x_max = min(x + half + 1, img.shape[1])
    return img[y_min:y_max, x_min:x_max]

# Initialize new columns
for ch in channels_to_measure:
    valid_tracks[f'intensity_ch{ch}'] = np.nan

valid_tracks['pearson_0_1'] = np.nan  
valid_tracks['M1_0_1'] = np.nan
valid_tracks['M2_0_1'] = np.nan

# Iterate and assign values
for idx, row in valid_tracks.iterrows():
    frame = int(row['frame'])
    y, x = row['y'], row['x']

    for ch in channels_to_measure:
        intensity = map_coordinates(
            img_array[frame, ch], [[y], [x]], order=1, mode='reflect'
        )[0]
        valid_tracks.at[idx, f'intensity_ch{ch}'] = intensity

    # Extract patches
    patch_ch0 = extract_patch(img_array[frame, 0], y, x, size=10)
    patch_ch1 = extract_patch(img_array[frame, 1], y, x, size=10)

    # Pearson correlations
    valid_tracks.at[idx, 'pearson_0_1'] = safe_pearsonr(patch_ch0, patch_ch1)

    # Manders coefficients
    M1_0_1, M2_0_1 = manders_coeffs(patch_ch0, patch_ch1)

    valid_tracks.at[idx, 'M1_0_1'] = M1_0_1
    valid_tracks.at[idx, 'M2_0_1'] = M2_0_1


In [ ]:
# Generate the lysosome mask and measure distance to nearest lysosome

In [ ]:
save_track_movies(img_array, 
                  valid_tracks[['particle', 'frame', 'y', 'x']],
                  image_name=image_path.name,
                  output_dir=f"{image_path.stem}_valid_tracks")

save_track_movies_with_overlay(img_array, 
                               valid_tracks[['particle', 'frame', 'y', 'x']],
                               image_name=image_path.name,
                               output_dir=f"{image_path.stem}_valid_tracks")

## Write valid track to a CSV

In [ ]:
print(valid_tracks)

In [ ]:
#valid_tracks.to_csv(valid_track_fn, index=False)

## View valid tracks

# Combine CSV files into one

In [ ]:
import pandas as pd
import glob
import os

# Get all CSV file paths
#csv_files = glob.glob(os.path.join(images_dir, '*valid_tracks.csv'))

# Concatenate all CSV files
#df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

# Optionally, save the result
#df.to_csv(os.path.join(images_dir, 'all_valid_20250605.csv'), index=False)


In [ ]:
test = df.groupby(['image_name','particle']).sum()
print(test)